# Step 1 - Explore production, build the Analytical Base Table (ABT)

**The setup:** our data-science team works in a **dev sandbox** (`ML_FRAUD_DEV_SANDBOX`) that lives *inside the same account* as production (`ML_FRAUD_PRODUCTION`). We run as `ML_DEV_ROLE`.

**The rule that makes this safe:** the dev role can **read all of production** - so we build on real data - but **cannot write to it**. Nothing a data scientist does here can change production, and that's enforced by RBAC, not by convention or good intentions. We prove it in the first cell below.

**What a data scientist actually does first (this notebook):**
1. Confirm the boundary: read prod works, writing prod is rejected.
2. Explore the production data to understand it.
3. Apply a **selective transform** into a clean, labeled **analytical base table (ABT)** in the sandbox - the single input for feature engineering and training.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("USE ROLE ML_DEV_ROLE").collect()
# Deactivate secondary roles so we operate purely as the data scientist. Snowsight
# sessions default to all roles active; an inherited admin role would otherwise mask
# the governance boundary we demonstrate next.
session.sql("USE SECONDARY ROLES NONE").collect()
session.sql("USE WAREHOUSE CORTEX_CODE_WH").collect()
print("role:", session.get_current_role())

## 1. The governance boundary - read prod, but you cannot change it

Before building anything, let's make the environment separation concrete. As `ML_DEV_ROLE` we can freely **read** production, but any attempt to **write** it - update a row, add a table - is rejected by Snowflake's access control. This is the guardrail that lets data scientists develop against real production data with zero risk to it. The two write attempts below are *expected to fail*.

In [ ]:
# Prove it: reading prod is allowed; writing prod is blocked by RBAC (not convention).
session.sql("USE ROLE ML_DEV_ROLE").collect()
session.sql("USE SECONDARY ROLES NONE").collect()  # act only as the data scientist

n = session.sql("SELECT COUNT(*) AS C FROM ML_FRAUD_PRODUCTION.CURATED.TXN_EVENTS").collect()[0]["C"]
print(f"READ  prod.CURATED.TXN_EVENTS   -> allowed ({n:,} rows)\n")

blocked = [
    ("UPDATE a row in prod", "UPDATE ML_FRAUD_PRODUCTION.CURATED.TXN_EVENTS SET IS_LAUNDERING = IS_LAUNDERING WHERE FALSE"),
    ("ADD a table in prod",  "CREATE TABLE ML_FRAUD_PRODUCTION.CURATED.DS_SCRATCH (id INT)"),
]
for label, sql in blocked:
    try:
        session.sql(sql).collect()
        print(f"WRITE {label:22} -> UNEXPECTEDLY ALLOWED (check grants!)")
    except Exception as e:
        print(f"WRITE {label:22} -> BLOCKED (by design): {str(e).splitlines()[0][:110]}")

## 2. Explore production data (read-only)

Now the everyday part: the DS reads production freely to understand it. Same role, same session - reads just work.

In [ ]:
-- Class balance + date window of the production events
SELECT COUNT(*) AS n_txns,
       SUM(IS_LAUNDERING) AS n_laundering,
       ROUND(100.0*SUM(IS_LAUNDERING)/COUNT(*), 3) AS laundering_pct,
       MIN(EVENT_TS) AS first_ts, MAX(EVENT_TS) AS last_ts
FROM ML_FRAUD_PRODUCTION.CURATED.TXN_EVENTS

In [ ]:
-- Laundering rate by payment channel (why ACH matters in this data)
SELECT PAYMENT_FORMAT,
       COUNT(*) AS n,
       SUM(IS_LAUNDERING) AS fraud,
       ROUND(100.0*SUM(IS_LAUNDERING)/COUNT(*), 4) AS fraud_pct
FROM ML_FRAUD_PRODUCTION.CURATED.TXN_EVENTS
GROUP BY PAYMENT_FORMAT ORDER BY n DESC

## 3. Selective transform -> the dev ABT

Notice the transform isn't written inline in this notebook - it lives in **`transforms/base_features.py`** and we just call `build_abt()`. That's deliberate, and it's the MLOps point:

- **The same function builds the prod ABT during promotion.** This exact code is re-run server-side when we deploy, so dev and production derive features from **identical logic** - no copy-pasted SQL drifting between environments.
- **It's a versioned, testable artifact**, not a cell. It goes through git and code review like any other production code; the notebook stays about *narrative and exploration*.

Here the DS builds the **dev** ABT from prod data: read production, write the sandbox.

In [ ]:
import sys, os
# git-backed workspace: repo root is the working dir; make transforms importable
for p in (os.getcwd(), os.path.dirname(os.getcwd())):
    if p not in sys.path:
        sys.path.insert(0, p)
from transforms.base_features import build_abt

abt = build_abt(session, env="dev")
print("Built ABT:", abt)

In [ ]:
-- Inspect the ABT the feature views will be built on
SELECT SPLIT, COUNT(*) AS n, SUM(IS_LAUNDERING) AS positives
FROM ML_FRAUD_DEV_SANDBOX.CURATED.FRAUD_ABT
GROUP BY SPLIT ORDER BY MIN(EVENT_TS)

## Next
The ABT (`ML_FRAUD_DEV_SANDBOX.CURATED.FRAUD_ABT`) is now the single input for the
dev feature store and model training. Continue with the feature-store setup, then
train and track an experiment - all in the dev sandbox.